# Credit Card Fraud Detection: EDA + Training Walkthrough

This notebook demonstrates:
1. Exploratory data analysis on synthetic transaction data
2. Preprocessing pipeline walkthrough
3. Quick GAN + GNN training demo
4. Evaluation metrics visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.synthetic_generator import SyntheticFraudGenerator
from preprocessing.cleaner import DataCleaner
from preprocessing.feature_engineering import FeatureEngineer

%matplotlib inline
plt.style.use('ggplot')
print('Imports OK')

## 1. Generate Synthetic Data

In [ ]:
gen = SyntheticFraudGenerator(n_samples=10_000, fraud_ratio=0.017, random_seed=42)
df = gen.generate()

print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["is_fraud"].mean():.3%}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Amount distribution
axes[0].hist(df[df['is_fraud']==0]['amount'], bins=50, alpha=0.7, label='Legit', color='steelblue')
axes[0].hist(df[df['is_fraud']==1]['amount'], bins=50, alpha=0.7, label='Fraud', color='crimson')
axes[0].set_xlabel('Amount ($)')
axes[0].set_title('Transaction Amount Distribution')
axes[0].legend()

# Hour of day
legit_hours = df[df['is_fraud']==0]['hour_of_day'].value_counts().sort_index()
fraud_hours = df[df['is_fraud']==1]['hour_of_day'].value_counts().sort_index()
axes[1].bar(legit_hours.index, legit_hours.values / legit_hours.values.sum(), alpha=0.7, label='Legit', color='steelblue')
axes[1].bar(fraud_hours.index, fraud_hours.values / fraud_hours.values.sum(), alpha=0.7, label='Fraud', color='crimson')
axes[1].set_xlabel('Hour of Day')
axes[1].set_title('Transaction Hour Distribution')
axes[1].legend()

# International vs domestic
intl_fraud = df.groupby('is_international')['is_fraud'].mean()
axes[2].bar(intl_fraud.index.map({0: 'Domestic', 1: 'International'}), intl_fraud.values, color=['steelblue', 'crimson'])
axes[2].set_ylabel('Fraud Rate')
axes[2].set_title('Fraud Rate by Transaction Type')

plt.tight_layout()
plt.show()

## 3. Preprocessing Pipeline

In [ ]:
cleaner = DataCleaner(outlier_method='iqr', outlier_threshold=3.0)
df_clean = cleaner.fit_transform(df)

engineer = FeatureEngineer(scaler_type='robust', time_windows=[])
df_feat = engineer.fit_transform(df_clean)

print(f'Feature columns: {len(engineer.feature_columns)}')
print(f'Sample features:\n{engineer.feature_columns[:10]}')

## 4. Quick Model Demo

In [ ]:
import torch
from models.gan.generator import Generator
from models.gan.discriminator import Discriminator

feature_dim = len(engineer.feature_columns)
gen = Generator(noise_dim=32, output_dim=feature_dim, hidden_dims=[64, 128, 64])
disc = Discriminator(input_dim=feature_dim, hidden_dims=[64, 128, 64])

# Generate samples
z = gen.sample_noise(8, torch.device('cpu'))
fake = gen(z)
scores = disc(fake)

print(f'Generator output shape: {fake.shape}')
print(f'Discriminator scores shape: {scores.shape}')

## 5. Evaluation Metrics Demo

In [ ]:
from evaluation.metrics import evaluate_model

# Simulate some predictions
np.random.seed(42)
n = 1000
y_true = (np.random.random(n) < 0.05).astype(int)
y_prob = np.clip(y_true * 0.7 + np.random.random(n) * 0.3, 0, 1)
y_pred = (y_prob > 0.5).astype(int)

metrics = evaluate_model(y_true, y_pred, y_prob)
print('Key metrics:')
for k in ['f1_weighted', 'roc_auc', 'auprc', 'mcc']:
    print(f'  {k}: {metrics.get(k, "N/A"):.4f}')